## Exploring Cold Start Problem in recommender systems. 
What items (books) should be recommended for a new user with no previous user history?

In [ ]:
# Standard library imports
import gzip
import html
import json
from pathlib import Path
import re

# Core third-party data & utility libraries
import numpy as np
import pandas as pd
from tqdm import tqdm

# Machine learning, NLP, and LSH libraries
from datasketch import MinHash, MinHashLSH
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df = pd.read_parquet("data/books_50pct_users_reviews.parquet")
meta = pd.read_parquet("data/books_50pct_users_meta.parquet")

In [4]:
meta.sample(n=2)

,item_id,title,description,price,brand,categories
6637,0061154237,"Fat, Broke &amp; Lonely No More: Your Personal...","&#8220;Let Victoria Moran be your personal, po...",11.04,,[[Books]]
185625,1433531585,Wondrous Works of God: A Family Bible Story Book,Starr Meadeserved as director of children&rsqu...,21.98,,[[Books]]


In [5]:
df.head(2)

,user_id,item_id,rating,summary,review_text,helpful_votes,timestamp
0,A10000012B7CGYKOMPQ4L,000100039X,5.0,Wonderful!,Spiritually and mentally inspiring! A book tha...,"[0, 0]",1355616000
1,A12387207U8U24,000100039X,5.0,Graet Work,"As you read, Gibran's poetry brings spiritual ...","[0, 0]",1206662400


### 1. Clean the complete metadata for the cold start problem: prepare the title, description and categories fields as one clean text per book. 

In [ ]:
# Input metadata parquet file
RAW_META_PARQUET = Path("data/books_5pct_users_meta.parquet")
# Cleaned output parquet file for downstream embedding models
OUTPUT_CLEAN_META = Path("data/books_clean_text_metadata.parquet")


def is_non_empty(val) -> bool:
    """Safely checks if a cell value is non-empty across strings, lists, dicts, arrays, and NaNs."""
    if val is None:
        return False
    if isinstance(val, (list, tuple, np.ndarray, set, dict)):
        return len(val) > 0
    try:
        if pd.isna(val):
            return False
    except (ValueError, TypeError):
        pass
    return len(str(val).strip()) > 0


def clean_html_and_text(text) -> str:
    """Removes HTML tags, unescapes HTML entities, and strips extra whitespace."""
    if text is None:
        return ""
    try:
        if pd.isna(text):
            return ""
    except (ValueError, TypeError):
        pass

    text_str = str(text)
    text_str = html.unescape(text_str)
    text_str = re.sub(r"<[^>]+>", " ", text_str)
    text_str = re.sub(r"\s+", " ", text_str).strip()
    return text_str


def extract_text_from_container(value) -> str:
    """Safely extracts and joins text from string, list of strings, or nested structures."""
    if value is None:
        return ""
    if isinstance(value, (list, tuple, np.ndarray, set)):
        flattened = []
        for item in value:
            if isinstance(item, (list, tuple, np.ndarray, set)):
                flattened.extend([clean_html_and_text(x) for x in item])
            else:
                flattened.append(clean_html_and_text(item))
        return " ".join([x for x in flattened if x])
    try:
        if pd.isna(value):
            return ""
    except (ValueError, TypeError):
        pass

    return clean_html_and_text(value)


def parse_details(details) -> dict:
    """Extracts publisher, page count, and author from dictionary or JSON string."""
    if details is None:
        return {"publisher": "", "page_count": "", "author": ""}

    if isinstance(details, str):
        try:
            details = json.loads(details)
        except Exception:
            details = {}

    if not isinstance(details, dict):
        return {"publisher": "", "page_count": "", "author": ""}

    normalized = {str(k).lower().strip(): str(v) for k, v in details.items()}

    publisher = (
        normalized.get("publisher")
        or normalized.get("imprint")
        or ""
    )
    pages = (
        normalized.get("page count")
        or normalized.get("print length")
        or normalized.get("paperback")
        or normalized.get("hardcover")
        or ""
    )
    author = (
        normalized.get("author")
        or normalized.get("by (author)")
        or ""
    )

    return {
        "publisher": clean_html_and_text(publisher),
        "page_count": clean_html_and_text(pages),
        "author": clean_html_and_text(author),
    }


def process_metadata():
    print(f"=== Step 1: Loading Parquet Metadata from {RAW_META_PARQUET} ===")
    df = pd.read_parquet(RAW_META_PARQUET)
    total_rows = len(df)
    print(f"Loaded {total_rows:,} rows.")

    # Standardize item ID column name
    id_col = "item_id" if "item_id" in df.columns else "parent_asin"
    if id_col not in df.columns:
        raise KeyError(f"Neither 'item_id' nor 'parent_asin' found in {df.columns.tolist()}")

    # ----------------------------------------------------
    # Completeness Check
    # ----------------------------------------------------
    print("\n--- Field Completeness Report ---")
    check_cols = ["title", "description", "features", "details", "categories", "brand", "author"]
    for col in check_cols:
        if col in df.columns:
            non_empty = df[col].apply(is_non_empty).sum()
            pct = (non_empty / total_rows) * 100
            print(f"  - {col:<12}: {non_empty:,} / {total_rows:,} ({pct:.2f}%)")
        else:
            print(f"  - {col:<12}: Column not present in Parquet file")

    # ----------------------------------------------------
    # Text Cleaning & Unification
    # ----------------------------------------------------
    print("\n--- Building Clean Unified Text ---")
    clean_records = []

    for _, row in tqdm(df.iterrows(), total=total_rows, desc="Cleaning Metadata"):
        item_id = str(row[id_col])

        # Clean individual components
        title_clean = clean_html_and_text(row.get("title", ""))
        desc_clean = extract_text_from_container(row.get("description", ""))
        feat_clean = extract_text_from_container(row.get("features", ""))
        cats_clean = extract_text_from_container(row.get("categories", ""))
        brand_clean = clean_html_and_text(row.get("brand", ""))

        # Parse nested details or direct author column
        details_parsed = parse_details(row.get("details", {}))
        author_clean = clean_html_and_text(row.get("author", "")) or details_parsed["author"]
        publisher_clean = clean_html_and_text(row.get("publisher", "")) or details_parsed["publisher"]
        pages_clean = clean_html_and_text(row.get("page_count", "")) or details_parsed["page_count"]

        # Build one comprehensive text representation per book
        text_parts = []
        if title_clean:
            text_parts.append(f"Title: {title_clean}")
        if author_clean:
            text_parts.append(f"Author: {author_clean}")
        if desc_clean:
            text_parts.append(f"Description: {desc_clean}")
        if feat_clean:
            text_parts.append(f"Features: {feat_clean}")
        if cats_clean:
            text_parts.append(f"Categories: {cats_clean}")
        if brand_clean:
            text_parts.append(f"Brand: {brand_clean}")
        if publisher_clean:
            text_parts.append(f"Publisher: {publisher_clean}")
        if pages_clean:
            text_parts.append(f"Pages: {pages_clean}")

        combined_text = " | ".join(text_parts)

        clean_records.append({
            "item_id": item_id,
            "title": title_clean,
            "author": author_clean,
            "publisher": publisher_clean,
            "combined_text": combined_text,
        })

    # Save cleaned metadata
    clean_df = pd.DataFrame(clean_records).drop_duplicates("item_id")
    clean_df.to_parquet(OUTPUT_CLEAN_META, index=False)
    print(f"\nSaved cleaned text metadata to '{OUTPUT_CLEAN_META}' ({len(clean_df):,} unique books).")


if __name__ == "__main__":
    process_metadata()

=== Step 1: Loading Parquet Metadata from data/books_5pct_users_meta.parquet ===
Loaded 349,741 rows.

--- Field Completeness Report ---
  - title       : 272,173 / 349,741 (77.82%)
  - description : 226,976 / 349,741 (64.90%)
  - features    : Column not present in Parquet file
  - details     : Column not present in Parquet file
  - categories  : 349,741 / 349,741 (100.00%)
  - brand       : 18 / 349,741 (0.01%)
  - author      : Column not present in Parquet file

--- Building Clean Unified Text ---


Cleaning Metadata: 100%|██████████| 349741/349741 [00:30<00:00, 11376.56it/s]



Saved cleaned text metadata to 'data/books_clean_text_metadata.parquet' (349,741 unique books).


### 2. Vector representation of the data. Embeddings using 2 methods, TF-IDF and SBERT. 

In [ ]:
INPUT_PARQUET = Path("data/books_clean_text_metadata.parquet")
OUT_TFIDF_SVD = Path("data/books_tfidf_svd_embeddings.npy")
OUT_SBERT = Path("data/books_sbert_embeddings.npy")


def generate_vectors():
    df = pd.read_parquet(INPUT_PARQUET)
    texts = df["combined_text"].fillna("").tolist()
    print(f"Loaded {len(df):,} items for vector generation.")

    # ----------------------------------------------------
    # Baseline: TF-IDF + TruncatedSVD (128 dimensions)
    # ----------------------------------------------------
    print("\n1. Building Baseline: TF-IDF + SVD (128d)...")
    tfidf = TfidfVectorizer(max_features=50_000, stop_words="english", ngram_range=(1, 2))
    tfidf_mat = tfidf.fit_transform(texts)

    svd = TruncatedSVD(n_components=128, random_state=42)
    svd_embeddings = svd.fit_transform(tfidf_mat)
    
    explained_var = svd.explained_variance_ratio_.sum() * 100
    print(f"   TF-IDF + SVD Matrix Shape: {svd_embeddings.shape}")
    print(f"   Explained Variance: {explained_var:.2f}%")
    np.save(OUT_TFIDF_SVD, svd_embeddings)

    # ----------------------------------------------------
    # Main Method: SBERT (all-MiniLM-L6-v2) (384 dimensions)
    # ----------------------------------------------------
    print("\n2. Building Main Method: SBERT (all-MiniLM-L6-v2)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    # Encode texts in mini-batches
    sbert_embeddings = model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True  # Cosine similarity equals inner product
    )

    print(f"   SBERT Matrix Shape: {sbert_embeddings.shape}")
    np.save(OUT_SBERT, sbert_embeddings)

    print(f"\nSaved vectors:\n  - Baseline: {OUT_TFIDF_SVD}\n  - SBERT:    {OUT_SBERT}")


if __name__ == "__main__":
    generate_vectors()

/Users/evelinahaite/Desktop/DTU_Masters/Term3/computational-tools-ds/02807_computational_tools_ds/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 10:11:50.787000 8406 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loaded 349,741 items for vector generation.

1. Building Baseline: TF-IDF + SVD (128d)...
   TF-IDF + SVD Matrix Shape: (349741, 128)
   Explained Variance: 20.88%

2. Building Main Method: SBERT (all-MiniLM-L6-v2)...


Batches: 100%|██████████| 1367/1367 [47:09<00:00,  2.07s/it] 


   SBERT Matrix Shape: (349741, 384)

Saved vectors:
  - Baseline: data/books_tfidf_svd_embeddings.npy
  - SBERT:    data/books_sbert_embeddings.npy


## Hashing to detect duplicates

In [ ]:
INPUT_PARQUET = Path("data/books_clean_text_metadata.parquet")
OUTPUT_DUPLICATES = Path("data/book_duplicate_clusters.parquet")


def get_shingles(text: str, k: int = 3) -> set:
    """Generates character k-shingles for string matching."""
    text = text.lower().strip()
    if len(text) < k:
        return {text}
    return {text[i : i + k] for i in range(len(text) - k + 1)}


def run_lsh_deduplication():
    print("=== Step 5: MinHash / LSH Duplicate Detection ===")
    df = pd.read_parquet(INPUT_PARQUET)

    # Create composite key for duplicate detection
    df["dedup_key"] = (df["title"].fillna("") + " " + df["author"].fillna("")).str.strip()
    df = df[df["dedup_key"] != ""].copy()

    # Initialize LSH index (Threshold = 0.85 Jaccard Similarity)
    lsh = MinHashLSH(threshold=0.85, num_perm=128)
    minhashes = {}

    print("Building MinHashes...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="MinHashing"):
        item_id = row["item_id"]
        shingles = get_shingles(row["dedup_key"])

        m = MinHash(num_perm=128)
        for s in shingles:
            m.update(s.encode("utf8"))

        minhashes[item_id] = m
        lsh.insert(item_id, m)

    # Query duplicate clusters
    print("Querying LSH index for duplicate clusters...")
    visited = set()
    clusters = []

    for item_id, mhash in tqdm(minhashes.items(), desc="Finding Clusters"):
        if item_id in visited:
            continue

        neighbors = lsh.query(mhash)
        if len(neighbors) > 1:
            canonical_id = neighbors[0]
            for n in neighbors:
                visited.add(n)
                clusters.append({"item_id": n, "canonical_item_id": canonical_id})

    cluster_df = pd.DataFrame(clusters)
    print(f"Found {len(cluster_df):,} entries mapping to duplicate groups.")

    cluster_df.to_parquet(OUTPUT_DUPLICATES, index=False)
    print(f"Saved duplicate mapping to {OUTPUT_DUPLICATES}")


if __name__ == "__main__":
    run_lsh_deduplication()

=== Step 5: MinHash / LSH Duplicate Detection ===
Building MinHashes...


MinHashing: 100%|██████████| 272173/272173 [02:29<00:00, 1821.07it/s]


Querying LSH index for duplicate clusters...


Finding Clusters: 100%|██████████| 272173/272173 [00:02<00:00, 106962.83it/s]


Found 18,496 entries mapping to duplicate groups.
Saved duplicate mapping to data/book_duplicate_clusters.parquet
